In [ ]:
from datetime import date
from enum import Enum
from typing import Self
from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator
import re

In [ ]:



class Coupon(BaseModel):
    """A discount coupon. `discount_percent` must be a multiple of 5."""

    model_config = ConfigDict(populate_by_name=True)
    
    code: str = Field(pattern=r"SAVE([0-9][0-9]?|100)$")
    discount_percent: int = Field(gt=4)
    
    @field_validator("discount_percent")
    @classmethod
    def discount_multiple_of_5(cls, discount_percent: int) -> int:
        if discount_percent%5 !=0:
            raise ValueError("discount_percent must be a multiple of 5")
        return discount_percent
    
    @model_validator(mode="after")
    def code_discount_match(self) -> Self:
        prcnt = self.discount_percent
        cde = self.code
        cde_int = int(re.search(r'\d+', cde).group())
        if cde_int != prcnt:
            raise ValueError("code and discount mismatch")

        return self

In [ ]:
try:
    discs=Coupon(code="SAVE10", discount_percent=15)
except Exception as e:
    print(e)

In [ ]:
class Booking(BaseModel):
    """A hotel booking. `check_out` must be after `check_in`."""
    guest_name: str
    check_in: date
    check_out:date

    
    @model_validator(mode="after")
    def checkout_validation(self) -> Self:
        if self.check_in >=self.check_out:
            raise ValueError("check_out must be after check_in")
        return self
        
        
    

In [ ]:
try:
    chckout = Booking( guest_name="Carla",
                      check_in=date(2027, 5, 1),
                      check_out=date(2026, 5, 5))
except Exception as e:
    print(e)

In [ ]:
chckout